# Pretrained Image Encoders: CLIP, DINOv2, and SigLIP

This notebook compares four approaches to image classification in Ludwig:

1. **stacked_cnn** — a convolutional network trained from scratch
2. **DINOv2 linear probe** — pretrained DINOv2 backbone, frozen; only the head is trained
3. **DINOv2 fine-tuned** — pretrained DINOv2 backbone, full fine-tuning
4. **CLIP** — pretrained CLIP vision encoder, frozen
5. **SigLIP** — pretrained SigLIP vision encoder, frozen

We use the [`beans`](https://huggingface.co/datasets/beans) dataset: a plant disease dataset with 3 classes (~1000 images total), which makes it a realistic small-data scenario where pretrained encoders shine.

**Runtime**: GPU is required for DINOv2/CLIP/SigLIP. In Colab, go to **Runtime → Change runtime type → T4 GPU**.

## Setup

In [ ]:
!pip install ludwig[vision] datasets --quiet

In [ ]:
import os
import time
import warnings

import torch

warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Dataset

The `beans` dataset contains RGB images of bean leaves classified into three categories:
- `angular_leaf_spot` — fungal disease
- `bean_rust` — rust disease  
- `healthy` — healthy leaf

It has ~1034 training images, 133 validation images, and 128 test images. This makes it a realistic small-data scenario.

In [ ]:
import csv
from pathlib import Path

from datasets import load_dataset

# Download the beans dataset from HuggingFace
ds = load_dataset("beans")
print(ds)

# Save images to disk and create CSV files
DATA_DIR = Path("/tmp/beans")
DATA_DIR.mkdir(parents=True, exist_ok=True)

label_names = ds["train"].features["labels"].names
print(f"Classes: {label_names}")


def save_split(split_name: str) -> str:
    """Save images to disk and return path to a CSV with image_path and label columns."""
    split_dir = DATA_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    csv_path = DATA_DIR / f"{split_name}.csv"

    rows = []
    for i, example in enumerate(ds[split_name]):
        img = example["image"]
        label_idx = example["labels"]
        label = label_names[label_idx]

        img_path = split_dir / f"{i:04d}_{label}.jpg"
        if not img_path.exists():
            img.save(str(img_path))

        rows.append({"image_path": str(img_path), "label": label})

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image_path", "label"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved {len(rows)} examples to {csv_path}")
    return str(csv_path)


train_csv = save_split("train")
val_csv = save_split("validation")
test_csv = save_split("test")

print(f"\nTrain: {train_csv}")
print(f"Val:   {val_csv}")
print(f"Test:  {test_csv}")

In [ ]:
# Preview a few examples
import pandas as pd

train_df = pd.read_csv(train_csv)
print(train_df.head())
print(f"\nClass distribution:\n{train_df['label'].value_counts()}")

In [ ]:
# Visualize a few images
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, (label, group) in enumerate(train_df.groupby("label")):
    sample = group.sample(1).iloc[0]
    img = Image.open(sample["image_path"])
    axes[i].imshow(img)
    axes[i].set_title(label)
    axes[i].axis("off")
plt.suptitle("Beans dataset — one example per class", fontsize=13)
plt.tight_layout()
plt.show()

## Helper: timing and memory tracking

In [ ]:
import gc

results = {}  # Will accumulate {encoder_name: {accuracy, train_time_s, peak_gpu_mb}}


def reset_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def peak_gpu_mb() -> float:
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**2
    return 0.0


def evaluate_model(model, test_csv: str) -> float:
    """Return accuracy on the test set."""
    eval_stats, _, _ = model.evaluate(dataset=test_csv, collect_overall_stats=True)
    return eval_stats.get("label", {}).get("accuracy", float("nan"))

## Approach 1: Stacked CNN (trained from scratch)

The `stacked_cnn` encoder is Ludwig's default: a stack of 2D convolutional layers followed by fully connected layers. There are no pretrained weights — the model learns everything from the 1034 training images.

This is the baseline. Expect lower accuracy and longer training time compared to pretrained encoders on this small dataset.

In [ ]:
from ludwig.api import LudwigModel

config_stacked_cnn = {
    "model_type": "ecd",
    "input_features": [
        {
            "name": "image_path",
            "type": "image",
            "encoder": {
                "type": "stacked_cnn",
                "use_pretrained": False,
            },
        }
    ],
    "output_features": [{"name": "label", "type": "category"}],
    "trainer": {"epochs": 20, "learning_rate": 0.001},
}

reset_gpu()
t0 = time.time()

model_cnn = LudwigModel(config=config_stacked_cnn, logging_level=30)
_, _, output_dir = model_cnn.train(
    dataset=train_csv,
    validation_set=val_csv,
    output_directory="/tmp/results/stacked_cnn",
    skip_save_processed_input=True,
)

train_time = time.time() - t0
gpu_mem = peak_gpu_mb()
accuracy = evaluate_model(model_cnn, test_csv)

results["stacked_cnn"] = {"accuracy": accuracy, "train_time_s": train_time, "peak_gpu_mb": gpu_mem}
print(f"stacked_cnn — accuracy: {accuracy:.4f}, time: {train_time:.1f}s, peak GPU: {gpu_mem:.0f}MB")

## Approach 2: DINOv2 linear probe

DINOv2 (Oquab et al., TMLR 2024) is a self-supervised vision transformer from Meta trained on 142M images. It produces rich general-purpose features without requiring image-text pairs.

In **linear probe** mode (`trainable: false`), the DINOv2 backbone is frozen — only the Ludwig output head (a small linear layer) is trained. This is:
- **Fast**: no backprop through the large backbone
- **Data-efficient**: works well with very few labeled examples
- **Memory-efficient**: no gradients stored for the backbone

Default model: `facebook/dinov2-base` (86M parameters, 768-dim output)

In [ ]:
config_dinov2_probe = {
    "model_type": "ecd",
    "input_features": [
        {
            "name": "image_path",
            "type": "image",
            "encoder": {
                "type": "dinov2",
                "use_pretrained": True,
                "trainable": False,  # freeze the backbone
            },
        }
    ],
    "output_features": [{"name": "label", "type": "category"}],
    "trainer": {"epochs": 10, "learning_rate": 0.001},
}

reset_gpu()
t0 = time.time()

model_dinov2_probe = LudwigModel(config=config_dinov2_probe, logging_level=30)
_, _, output_dir = model_dinov2_probe.train(
    dataset=train_csv,
    validation_set=val_csv,
    output_directory="/tmp/results/dinov2_linear_probe",
    skip_save_processed_input=True,
)

train_time = time.time() - t0
gpu_mem = peak_gpu_mb()
accuracy = evaluate_model(model_dinov2_probe, test_csv)

results["dinov2_linear_probe"] = {"accuracy": accuracy, "train_time_s": train_time, "peak_gpu_mb": gpu_mem}
print(f"DINOv2 linear probe — accuracy: {accuracy:.4f}, time: {train_time:.1f}s, peak GPU: {gpu_mem:.0f}MB")

## Approach 3: DINOv2 fine-tuned

In **fine-tuning** mode (`trainable: true`), gradients flow through the entire DINOv2 backbone. This can improve accuracy over linear probing but requires:
- More GPU memory (stores activations for backprop through 86M parameters)
- A lower learning rate (to avoid destroying pretrained features)
- Fewer epochs (the model has a strong starting point)

Use fine-tuning when you have enough data and GPU memory and want maximum accuracy.

In [ ]:
config_dinov2_finetune = {
    "model_type": "ecd",
    "input_features": [
        {
            "name": "image_path",
            "type": "image",
            "encoder": {
                "type": "dinov2",
                "use_pretrained": True,
                "trainable": True,  # full fine-tuning
            },
        }
    ],
    "output_features": [{"name": "label", "type": "category"}],
    "trainer": {
        "epochs": 5,
        "learning_rate": 0.0001,  # lower LR for fine-tuning
    },
}

reset_gpu()
t0 = time.time()

model_dinov2_ft = LudwigModel(config=config_dinov2_finetune, logging_level=30)
_, _, output_dir = model_dinov2_ft.train(
    dataset=train_csv,
    validation_set=val_csv,
    output_directory="/tmp/results/dinov2_finetuned",
    skip_save_processed_input=True,
)

train_time = time.time() - t0
gpu_mem = peak_gpu_mb()
accuracy = evaluate_model(model_dinov2_ft, test_csv)

results["dinov2_finetuned"] = {"accuracy": accuracy, "train_time_s": train_time, "peak_gpu_mb": gpu_mem}
print(f"DINOv2 fine-tuned — accuracy: {accuracy:.4f}, time: {train_time:.1f}s, peak GPU: {gpu_mem:.0f}MB")

## Approach 4: CLIP

CLIP (Radford et al., ICML 2021) from OpenAI is trained on 400M image-text pairs using contrastive learning. Its visual encoder produces embeddings that are aligned with text in a shared latent space.

CLIP features are particularly useful for:
- Zero-shot image classification (not shown here)
- Tasks where visual-semantic alignment matters
- Multimodal applications

Default model: `openai/clip-vit-base-patch32` (86M parameters, 768-dim output)

In [ ]:
config_clip = {
    "model_type": "ecd",
    "input_features": [
        {
            "name": "image_path",
            "type": "image",
            "encoder": {
                "type": "clip",
                "use_pretrained": True,
                "trainable": False,
            },
        }
    ],
    "output_features": [{"name": "label", "type": "category"}],
    "trainer": {"epochs": 10, "learning_rate": 0.001},
}

reset_gpu()
t0 = time.time()

model_clip = LudwigModel(config=config_clip, logging_level=30)
_, _, output_dir = model_clip.train(
    dataset=train_csv,
    validation_set=val_csv,
    output_directory="/tmp/results/clip",
    skip_save_processed_input=True,
)

train_time = time.time() - t0
gpu_mem = peak_gpu_mb()
accuracy = evaluate_model(model_clip, test_csv)

results["clip"] = {"accuracy": accuracy, "train_time_s": train_time, "peak_gpu_mb": gpu_mem}
print(f"CLIP linear probe — accuracy: {accuracy:.4f}, time: {train_time:.1f}s, peak GPU: {gpu_mem:.0f}MB")

## Approach 5: SigLIP

SigLIP (Zhai et al., ICCV 2023) from Google uses sigmoid loss instead of softmax contrastive loss for image-text pre-training. This removes the dependency on global batch statistics and enables better scaling.

SigLIP features are similar to CLIP in nature but often outperform CLIP on downstream classification tasks, especially with smaller models.

Default model: `google/siglip-base-patch16-224` (86M parameters, 768-dim output)

In [ ]:
config_siglip = {
    "model_type": "ecd",
    "input_features": [
        {
            "name": "image_path",
            "type": "image",
            "encoder": {
                "type": "siglip",
                "use_pretrained": True,
                "trainable": False,
            },
        }
    ],
    "output_features": [{"name": "label", "type": "category"}],
    "trainer": {"epochs": 10, "learning_rate": 0.001},
}

reset_gpu()
t0 = time.time()

model_siglip = LudwigModel(config=config_siglip, logging_level=30)
_, _, output_dir = model_siglip.train(
    dataset=train_csv,
    validation_set=val_csv,
    output_directory="/tmp/results/siglip",
    skip_save_processed_input=True,
)

train_time = time.time() - t0
gpu_mem = peak_gpu_mb()
accuracy = evaluate_model(model_siglip, test_csv)

results["siglip"] = {"accuracy": accuracy, "train_time_s": train_time, "peak_gpu_mb": gpu_mem}
print(f"SigLIP linear probe — accuracy: {accuracy:.4f}, time: {train_time:.1f}s, peak GPU: {gpu_mem:.0f}MB")

## Results comparison

In [ ]:
import pandas as pd

rows = []
for name, metrics in results.items():
    rows.append(
        {
            "Encoder": name,
            "Accuracy": f"{metrics['accuracy']:.4f}",
            "Train time (s)": f"{metrics['train_time_s']:.1f}",
            "Peak GPU (MB)": f"{metrics['peak_gpu_mb']:.0f}" if metrics["peak_gpu_mb"] > 0 else "N/A",
        }
    )

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

In [ ]:
# Plot accuracy comparison
import matplotlib.pyplot as plt

names = list(results.keys())
accuracies = [results[n]["accuracy"] for n in names]
times = [results[n]["train_time_s"] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]

bars = ax1.bar(names, accuracies, color=colors[: len(names)])
ax1.set_ylim(0, 1.05)
ax1.set_ylabel("Test Accuracy")
ax1.set_title("Accuracy by encoder")
ax1.tick_params(axis="x", rotation=20)
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{acc:.3f}", ha="center", va="bottom")

bars2 = ax2.bar(names, times, color=colors[: len(names)])
ax2.set_ylabel("Training time (s)")
ax2.set_title("Training time by encoder")
ax2.tick_params(axis="x", rotation=20)
for bar, t in zip(bars2, times):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{t:.0f}s", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## Few-shot: 5 examples per class

One of the biggest advantages of pretrained encoders is their ability to generalize from very few labeled examples. In this section, we compare `stacked_cnn` vs `dinov2` (linear probe) when trained on only **5 examples per class** (15 examples total).

This simulates a realistic scenario where annotation is expensive.

In [ ]:
import csv
from pathlib import Path

import pandas as pd

# Build a 5-shot training set (5 examples per class)
train_df = pd.read_csv(train_csv)
fewshot_df = train_df.groupby("label").sample(n=5, random_state=42).reset_index(drop=True)

fewshot_csv = "/tmp/beans/fewshot_5shot.csv"
fewshot_df.to_csv(fewshot_csv, index=False)

print(f"Few-shot training set: {len(fewshot_df)} examples")
print(fewshot_df["label"].value_counts())

In [ ]:
fewshot_results = {}

# Few-shot with stacked_cnn
reset_gpu()
t0 = time.time()
model_cnn_fs = LudwigModel(config=config_stacked_cnn, logging_level=30)
model_cnn_fs.train(
    dataset=fewshot_csv,
    output_directory="/tmp/results/fewshot_cnn",
    skip_save_processed_input=True,
)
train_time = time.time() - t0
accuracy = evaluate_model(model_cnn_fs, test_csv)
fewshot_results["stacked_cnn (5-shot)"] = {"accuracy": accuracy, "train_time_s": train_time}
print(f"stacked_cnn (5-shot) — accuracy: {accuracy:.4f}, time: {train_time:.1f}s")

In [ ]:
# Few-shot with DINOv2 linear probe
reset_gpu()
t0 = time.time()
model_dinov2_fs = LudwigModel(config=config_dinov2_probe, logging_level=30)
model_dinov2_fs.train(
    dataset=fewshot_csv,
    output_directory="/tmp/results/fewshot_dinov2",
    skip_save_processed_input=True,
)
train_time = time.time() - t0
accuracy = evaluate_model(model_dinov2_fs, test_csv)
fewshot_results["dinov2_linear_probe (5-shot)"] = {"accuracy": accuracy, "train_time_s": train_time}
print(f"DINOv2 linear probe (5-shot) — accuracy: {accuracy:.4f}, time: {train_time:.1f}s")

In [ ]:
print("\nFew-shot results (5 examples per class):")
print("-" * 55)
print(f"{'Encoder':<35} {'Accuracy':>10} {'Time':>8}")
print("-" * 55)
for name, m in fewshot_results.items():
    print(f"{name:<35} {m['accuracy']:>10.4f} {m['train_time_s']:>7.1f}s")
print("-" * 55)

print("\nFull dataset results (for reference):")
print("-" * 55)
print(f"{'Encoder':<35} {'Accuracy':>10} {'Time':>8}")
print("-" * 55)
for name in ["stacked_cnn", "dinov2_linear_probe"]:
    m = results[name]
    print(f"{name:<35} {m['accuracy']:>10.4f} {m['train_time_s']:>7.1f}s")
print("-" * 55)

## Summary and recommendations

### When to use each encoder

| Scenario | Recommended encoder |
|---|---|
| Large dataset (>10k images), custom domain | `stacked_cnn` or fine-tuned pretrained |
| Small dataset (<1k images), natural images | `dinov2` linear probe |
| Few-shot (<50 examples total) | `dinov2` or `siglip` linear probe |
| Multimodal or image-text retrieval | `clip` or `siglip` |
| Best possible accuracy, GPU available | `dinov2` fine-tuned |

### Key takeaways

- **Linear probing is surprisingly effective**: freezing the pretrained backbone and training only the head takes a fraction of the time and memory while achieving strong accuracy on small datasets.
- **DINOv2 is the most versatile**: its self-supervised training (no text labels needed) makes it robust across diverse image domains.
- **CLIP and SigLIP are better for semantically rich tasks**: their image-text alignment gives them an edge when classes have meaningful visual-semantic structure.
- **Few-shot gap is dramatic**: with only 5 examples per class, pretrained encoders maintain high accuracy while the CNN from scratch struggles.

### Changing the pretrained model

You can use any HuggingFace-compatible model by overriding `pretrained_model_name_or_path`:

```python
{
    "type": "dinov2",
    "use_pretrained": True,
    "trainable": False,
    "pretrained_model_name_or_path": "facebook/dinov2-large",  # larger model
}
```

Available DINOv2 variants: `facebook/dinov2-small`, `facebook/dinov2-base`, `facebook/dinov2-large`, `facebook/dinov2-giant`

Available CLIP variants: `openai/clip-vit-base-patch16`, `openai/clip-vit-large-patch14`

Available SigLIP variants: `google/siglip-base-patch16-224`, `google/siglip-large-patch16-256`, `google/siglip-so400m-patch14-384`